# 🔐 Mafsal — Server Setup (Google Colab)

Run each cell **in order**, top to bottom. The whole setup takes about 60 seconds.

When Cell 4 shows `Server ready`, copy the `wss://` URL printed in Cell 3 and use it on the kiosk.

> **Keep this tab open.** Closing it stops the tunnel and the server.

---

## Cell 1 — Install dependencies
Downloads `websockets` and `cryptography` into the Colab runtime. Takes ~15 seconds.

In [ ]:
!pip install -q websockets cryptography
print('✓ Dependencies installed.')

## Cell 2 — Upload server files

Upload `mafsal_server.py` and `traffic_shaping.py` from your copy of the repo.

Use the **folder icon** in the left sidebar → click the upload button → select both files.

Then run this cell to confirm they arrived.

In [ ]:
import os

missing = [f for f in ['mafsal_server.py', 'traffic_shaping.py']
           if not os.path.exists(f'/content/{f}')]

if missing:
    raise FileNotFoundError(
        f'Missing files in /content/: {missing}\n'
        'Upload them via the sidebar folder icon before continuing.'
    )

print('✓ mafsal_server.py   found')
print('✓ traffic_shaping.py found')

## Cell 3 — Set your RELAY_KEY

**Replace `PASTE_YOUR_KEY_HERE` with the key you generated locally:**

```bash
python3 -c "import os; print(os.urandom(32).hex())"
```

The key must be at least 64 hexadecimal characters. Use the **same key** on the kiosk side.

> ⚠️ Do not share this key. It authenticates the connection.

In [ ]:
import os

# ─── EDIT THIS LINE ──────────────────────────────────────────────────────────
RELAY_KEY = 'PASTE_YOUR_KEY_HERE'
# ─────────────────────────────────────────────────────────────────────────────

if RELAY_KEY == 'PASTE_YOUR_KEY_HERE' or len(RELAY_KEY) < 64:
    raise ValueError(
        'RELAY_KEY is missing or too short.\n'
        'Generate one locally:\n'
        '  python3 -c "import os; print(os.urandom(32).hex())"\n'
        'Then paste it above (minimum 64 hex characters).'
    )

os.environ['RELAY_KEY'] = RELAY_KEY
print(f'✓ RELAY_KEY set: {RELAY_KEY[:8]}...{RELAY_KEY[-4:]}  ({len(RELAY_KEY)} chars)')

## Cell 4 — Install and start Cloudflare Tunnel

Downloads `cloudflared` (the Cloudflare tunnel binary) and starts a tunnel that points at
the Mafsal WebSocket server on port 8765.

After ~10 seconds you will see a box like:

```
══════════════════════════════════════════════════════════════
  ★ Use this as COLAB_WS_URL on the kiosk:
    wss://abcd-efgh-1234.trycloudflare.com/bridge
══════════════════════════════════════════════════════════════
```

**Copy that `wss://` URL.** You will need it in the next step on the kiosk.
The URL changes every time you restart Colab.

In [ ]:
import subprocess, sys, os, re, threading, time

# ── Download cloudflared ───────────────────────────────────────────────────
cf_path = '/usr/local/bin/cloudflared'
if not os.path.exists(cf_path):
    print('Downloading cloudflared...')
    subprocess.run([
        'wget', '-q', '-O', cf_path,
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
    ], check=True)
    os.chmod(cf_path, 0o755)
    print('✓ cloudflared downloaded')
else:
    print('✓ cloudflared already present')

# ── Start tunnel ───────────────────────────────────────────────────────────
cf_url = {'ws': None}

def run_tunnel():
    proc = subprocess.Popen(
        [cf_path, 'tunnel', '--url', 'http://localhost:8765'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in proc.stdout:
        print(line, end='')
        m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
        if m:
            url    = m.group(0)
            ws_url = url.replace('https://', 'wss://') + '/bridge'
            cf_url['ws'] = ws_url
            sep = '=' * 62
            print(f'\n{sep}')
            print(f'  ★ Use this as COLAB_WS_URL on the kiosk:')
            print(f'    {ws_url}')
            print(f'{sep}\n')

threading.Thread(target=run_tunnel, daemon=True).start()
print('Cloudflare Tunnel starting... (URL appears below in ~10 s)')

## Cell 5 — Start the Mafsal server ▶︎

This cell starts `mafsal_server.py` and blocks. **It must keep running** for the tunnel to work.

You should see:
```
HH:MM:SS [INFO] Server ready → ws://0.0.0.0:8765
```

Once you see that line, go to the kiosk and run `launch_zero_trust.sh`.

In [ ]:
import asyncio, os, sys
sys.path.insert(0, '/content')

# Sanity check
for f in ['mafsal_server.py', 'traffic_shaping.py']:
    assert os.path.exists(f'/content/{f}'), f'{f} not found in /content/'

assert os.environ.get('RELAY_KEY'), 'RELAY_KEY not set — run Cell 3 first'

exec(open('/content/mafsal_server.py').read())
asyncio.run(main())

## Cell 6 — (Optional) Connection self-test

Run this in a **separate browser tab** (File → New notebook, or duplicate this tab)
while Cell 5 is still running. It performs a full v4 handshake and sends one HTTP request
to verify the server is working correctly.

In [ ]:
import asyncio, websockets, json, base64, os, struct
import hmac as _hmac, hashlib
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric.x25519 import (
    X25519PrivateKey, X25519PublicKey,
)
from cryptography.hazmat.backends import default_backend

import sys
sys.path.insert(0, '/content')
from traffic_shaping import add_padding, strip_padding, DEFAULT_CONFIG

SHARED_KEY = bytearray(bytes.fromhex(os.environ['RELAY_KEY']))

def _derive(sk, salt, epoch, ctr, direction):
    info = f'mafsal-v4:{direction}:{epoch}:{ctr}'.encode()
    return HKDF(hashes.SHA256(), 32, salt, info, default_backend()).derive(bytes(sk))

def _encrypt(pt, sk, epoch, ctr, direction):
    padded = add_padding(pt, DEFAULT_CONFIG)
    salt   = os.urandom(8)
    key    = _derive(sk, salt, epoch, ctr, direction)
    nonce  = os.urandom(12)
    aad    = salt + struct.pack('>QQ', epoch, ctr)
    ct     = AESGCM(key).encrypt(nonce, padded, aad)
    return salt + struct.pack('>QQ', epoch, ctr) + nonce + ct

def _decrypt(data, sk, epoch, direction):
    salt    = data[:8]
    counter = struct.unpack('>Q', data[16:24])[0]
    nonce   = data[24:36]
    ct      = data[36:]
    aad     = salt + struct.pack('>QQ', epoch, counter)
    key     = _derive(sk, salt, epoch, counter, direction)
    padded  = AESGCM(key).decrypt(nonce, ct, aad)
    return strip_padding(padded, DEFAULT_CONFIG), counter

async def self_test():
    async with websockets.connect('ws://localhost:8765/bridge') as ws:
        # Handshake
        priv      = X25519PrivateKey.generate()
        pub_bytes = priv.public_key().public_bytes_raw()
        mac       = _hmac.new(bytes(SHARED_KEY), pub_bytes, hashlib.sha256).hexdigest()
        await ws.send(json.dumps({'v': 4, 'type': 'hello',
                                   'pub': base64.b64encode(pub_bytes).decode(),
                                   'mac': mac}))
        resp      = json.loads(await ws.recv())
        assert resp['type'] == 'epoch', f'Unexpected: {resp}'

        epoch       = int(resp['epoch'])
        epoch_bytes = struct.pack('>Q', epoch)
        srv_pub     = base64.b64decode(resp['pub'])
        expected    = _hmac.new(bytes(SHARED_KEY),
                                epoch_bytes + srv_pub, hashlib.sha256).hexdigest()
        assert _hmac.compare_digest(expected, resp['mac']), 'Server HMAC invalid!'

        shared      = priv.exchange(X25519PublicKey.from_public_bytes(srv_pub))
        session_key = bytearray(
            HKDF(hashes.SHA256(), 32, epoch_bytes, b'mafsal-v4-session',
                 default_backend()).derive(shared)
        )
        print(f'✓ Handshake OK  (epoch={epoch})')

        # Test request
        req = b'GET / HTTP/1.1\r\nHost: httpbin.org\r\nConnection: close\r\n\r\n'
        enc = _encrypt(req, session_key, epoch, 0, 'C2S')
        msg = json.dumps({'v': 4, 'data': base64.b64encode(enc).decode(), 'size': len(enc)})
        await ws.send(msg)

        raw2     = json.loads(await ws.recv())
        plain, _ = _decrypt(base64.b64decode(raw2['data']), session_key, epoch, 'S2C')
        print(f'✓ Response received: {len(plain)} bytes')
        print('✓ Self-test passed — server is working correctly.')

asyncio.run(self_test())

---

**Mafsal** — Copyright (C) 2026 Mafsal Contributors. GPL-3.0-or-later.

For full documentation see `README.md` and `TECHNICAL_REFERENCE.md` in the repository.

To report issues or submit a fork → see `CONTRIBUTING.md`.